# Capstone — the paper's source notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tessa-Saumu/FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**What this notebook is.** Weeks 4 to 7 built, tested, and turned into a playbook a way to rank web pages by how likely they are to lose search traffic next month. This notebook assembles that work into the research paper and checks it. Every section below mirrors a section of the deployed paper. Every number is loaded from a **receipt** — a committed JSON file written by an executed earlier notebook — and the last cell re-checks each headline number against its receipt before the paper ships. Nothing new is computed from the warehouse here and no new decision is made: the warehouse work lives in `w03`-`w07`, and this notebook runs anywhere, with or without warehouse access.

**Terms used throughout, defined once.**

- Page: one piece of content on one client's website. Pages and clients appear only as anonymous scrambled codes — nothing client-identifying is in the data.
- Impressions: how many times a page appeared in Google search results in a 30-day window. Clicks: how many of those turned into a visit. CTR: clicks divided by impressions, as a percentage.
- Position: the page's average rank in Google results. 1 is the top result.
- Decline: the label. A page's impressions in the next 30 days fall below 80% of its impressions in the last 30 days.
- Base rate: the share of all pages that declined. Any ranking has to be read against it.
- Precision@50 (P@50): of the 50 highest-ranked pages, the share that really declined. The metric, because the review budget is 50 pages a month.
- The rule: the Week-4 hand-written baseline score. The models: logistic regression (LR) and random forest (RF) from Week 5.
- Frame: one table, one row per page — a feature month plus the next month's outcome. The March frame is March features, April outcomes.
- Receipt: a committed JSON of numbers produced by an executed notebook. Every number in the paper traces to one.

**The paper in five lines.**

- Question: an editor can review only 50 pages a month — which 50 should be at the top of the list?
- Data: 95,810 pages across 40 client sites from the anonymized FlyRank internship warehouse (~79M daily rows); five numbers per page, all known at month-end.
- Result: inside the month the model beat the hand rule (P@50 0.636 vs 0.512, base rate 0.49). One month forward the edge was mostly gone (0.54 vs rule 0.44, April base rate 0.56).
- Decision supported: a monthly review order for a human editor, with a reason per page, gates, and a fall-back to the rule.
- Biggest limitation: the ranking decays within weeks and must be refit monthly; a naive random split would have shown 0.98 and been a lie.

**Where the paper lives.** The paper's source is `work/paper/index.html` — a single self-contained file (its figures are embedded), so nothing of this project lives outside `work/`. `work/scripts/deploy_paper.py` publishes it to a `gh-pages` branch, which GitHub Pages serves. When it is live, that exact URL goes in `submission/paper_url.txt` (one line, nothing else).


## 1. Question

*The research question and the decision it supports.*

Search traffic to a page rarely dies in one day; it slides. By the time a monthly report looks bad, the decline has usually been under way for weeks. So the standing question for a content team managing many sites is: **of all the pages we could look at this month, which 50 should a person actually spend time on?**

- **Decision:** the order of a monthly review queue. Not a yes/no flag for every page — a ranking, because the budget is fixed at 50 reviews a month.
- **Actor:** a content lead or editor running a monthly refresh sprint across many client sites.
- **Action:** the reviewer works down the queue, checking each page before changing anything.
- **Cost of a wrong call:** a wasted review burns about an hour of a ~50-hour monthly budget; a missed decline leaves a page to decay another month. Neither is catastrophic — which is why this is decision support for a human, not automation.

There is already a sensible hand answer (the Week-4 rule: old pages and top-10 pages with low CTR decline more often — both confirmed in the Week-4 signal audit). The problem is that it is blunt: **24,462 pages tie at the rule's maximum score** in the March frame, so "top 50 by the rule" is a lottery draw from a band whose own decline rate (50.7%) is barely above the month's base rate (49.1%). The model's one job is to order that band.

But "did the model beat the rule" turned out to be the wrong question on its own: the answer depends entirely on how the comparison is run. The paper's spine is that dependency — the same three scorers (rule, LR, RF) under three increasingly honest tests, ending with the deployment question: trained on one month, scored on the next.


In [1]:
# Load every receipt this notebook and the paper quote from.
# Local run: reads work/outputs/. Colab: downloads the committed copies from GitHub.
import json
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/Tessa-Saumu/FlyRank-ML-Internship/main/work/outputs/"

def load_receipt(name):
    local = os.path.join("work", "outputs", name)
    if not os.path.exists(local):
        local = os.path.join("receipts", name)
        os.makedirs("receipts", exist_ok=True)
        if not os.path.exists(local):
            print(f"downloading {name} from the repo ...")
            urllib.request.urlretrieve(REPO_RAW + name, local)
    with open(local) as f:
        return json.load(f)

val = load_receipt("validation_audit_metrics.json")   # w06: splits, walk-forward, leak test
model = load_receipt("model_metrics.json")            # w05: grouped folds, versions
base = load_receipt("baseline_metrics.json")          # w04: rule, tie band
play = load_receipt("action_playbook_summary.json")   # w07: queue, gates, monitoring

march, april = val["frames"]["march"], val["frames"]["april"]
print("Development frame (March 2026):", f"{march['rows']:,} pages,",
      f"{march['clients']} clients, base rate {march['base_rate']:.3f}")
print("Scoring frame   (April 2026):", f"{april['rows']:,} pages,",
      f"{april['clients']} clients, base rate {april['base_rate']:.3f}")
print("Seed:", val["seed"], "| run versions:", ", ".join(f"{k} {v}" for k, v in val["library_versions"].items()))

# The three tests, one line each (the paper's Table 2):
rand, grouped, fwd = val["random_split"], val["grouped_cv"]["summary"], val["time_forward"]
print("\nThe three tests, P@50 (rule / LR / RF | base rate):")
print(f"  A. random split : {rand['metrics']['baseline']['p50']:.2f} / {rand['metrics']['logistic_regression']['p50']:.2f}"
      f" / {rand['metrics']['random_forest']['p50']:.2f} | {rand['test_base_rate']:.2f}"
      f"  ({rand['shared_clients_both_sides']} of 40 clients on both sides)")
print(f"  B. client-grouped: {grouped['baseline']['p50_mean']:.3f} / {grouped['logistic_regression']['p50_mean']:.3f}"
      f" / {grouped['random_forest']['p50_mean']:.3f} | fold mean "
      f"{sum(f['test_base_rate'] for f in val['grouped_cv']['per_fold'])/5:.2f}")
print(f"  C. one month forward: {fwd['metrics']['baseline']['p50']:.2f} / {fwd['metrics']['logistic_regression']['p50']:.2f}"
      f" / {fwd['metrics']['random_forest']['p50']:.2f} | {fwd['test_base_rate']:.2f}")


downloading validation_audit_metrics.json from the repo ...
downloading model_metrics.json from the repo ...
downloading baseline_metrics.json from the repo ...
downloading action_playbook_summary.json from the repo ...
Development frame (March 2026): 95,810 pages, 40 clients, base rate 0.491
Scoring frame   (April 2026): 99,279 pages, 44 clients, base rate 0.560
Seed: 42 | run versions: pandas 3.0.3, numpy 2.5.1, scikit-learn 1.9.0, duckdb 1.5.4

The three tests, P@50 (rule / LR / RF | base rate):
  A. random split : 0.44 / 0.74 / 0.98 | 0.49  (34 of 40 clients on both sides)
  B. client-grouped: 0.512 / 0.636 / 0.608 | fold mean 0.49
  C. one month forward: 0.44 / 0.54 / 0.14 | 0.56


## 2. Data

*Which release, which tables, date windows, what was excluded and why. Public-safe.*

**Source.** The FlyRank ML Internship warehouse release, `FlyRank/internship-warehouse` on Hugging Face (gated; request access and accept the data-use terms, approval is instant): about **79 million daily rows** covering late 2025 through June 30, 2026, read with DuckDB, never downloaded in bulk. The release is public-safe by construction — **no client names, domains, URLs, page titles, or search queries exist in it**. Pages and clients appear only as scrambled codes, used for joining and grouping, never as model features.

**Tables used.** `fact_content_daily_performance` (one row per page per day of search performance), `dim_content` (page metadata, including creation date), `dim_clients` (client context). The query-level table was **excluded** because its fixed 90-day window can overlap the outcome month — future information.

**Frames.** Everything is built as monthly frames: features from one calendar month, the label from the next. March 2026 is the development frame (95,810 pages, 40 clients, features Mar 2-31, labels Apr 1-30, base rate 49.1%, largest client 22.1%). April 2026 is the frame scored for the queue (99,279 pages, 44 clients, features Apr 1-30, labels May 1-30, base rate 56.0%).

**Who is in, who is out.**

- Floors, applied upstream in the Week-3 data contract: at least **100 impressions** and **14 days of data** in the feature month. Below that, CTR and position are noise on tiny denominators and one quiet day can look like a decline.
- Outcome-month tracking: March kept 95,810 of 98,398 candidates (**2.6% dropped**: 391 with no outcome rows, 2,197 with thin coverage); April kept 99,279 of 102,943 (3.6% dropped). The bias points one way — pages whose tracking went quiet are excluded, so everything here describes pages that could still be measured.
- Excluded fields: all future-window values (next-month impressions are the answer, not an input), label-derived trend fields, provider/model production metadata, fixed-window query signals.

**The label.** A page is labelled declining when its next-30-day impressions fall below **80%** of its last-30-day impressions. The 0.80 line is a contracted policy choice, checked for sensitivity: decline rates of 0.426 at 0.70, 0.491 at 0.80, 0.556 at 0.90. June 2026 partitions were sealed and never read, so a future test exists if this work continues.


In [2]:
# Table 1 of the paper: the two working frames, straight from the w06 receipt.
import pandas as pd

rows = []
for name, fr in [("March 2026 (development)", val["frames"]["march"]),
                 ("April 2026 (scored for the queue)", val["frames"]["april"])]:
    rows.append({
        "frame": name,
        "feature window": f"{fr['feature_window'][0]} to {fr['feature_window'][1]}",
        "label window": f"{fr['label_window'][0]} to {fr['label_window'][1]}",
        "pages": fr["rows"],
        "clients": fr["clients"],
        "base rate": round(fr["base_rate"], 3),
        "largest client": round(fr["largest_client_share"], 3),
    })
display(pd.DataFrame(rows))

sv = val["survivorship"]
print("Survivorship (pages dropped because the outcome month could not measure them):")
for k, v in sv.items():
    print(f"  {k}: {v['labeled']:,} labelled of {v['pool_candidates']:,} candidates "
          f"(-{v['dropped_absent_outcome']} absent, -{v['dropped_thin_outcome']} thin)")
print("Tie band (March):", f"{base['tie_band']['n']:,} pages share the rule's maximum score;",
      "band decline rate", round(base["tie_band"]["decline_rate"], 3))


,frame,feature window,label window,pages,clients,base rate,largest client
0,March 2026 (development),2026-03-02 to 2026-03-31,2026-04-01 to 2026-04-30,95810,40,0.491,0.221
1,April 2026 (scored for the queue),2026-04-01 to 2026-04-30,2026-05-01 to 2026-05-30,99279,44,0.560,0.224


Survivorship (pages dropped because the outcome month could not measure them):
  march_to_april: 95,810 labelled of 98,398 candidates (-391 absent, -2197 thin)
  april_to_may: 99,279 labelled of 102,943 candidates (-506 absent, -3158 thin)
Tie band (March): 24,462 pages share the rule's maximum score; band decline rate 0.507


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions, stated plainly.** The label is a proxy (a 20% impressions drop), not a business outcome. At decision time only the feature month's five numbers and the page's age exist, so only they are inputs. Review capacity, not scoring capacity, is the constraint (50 pages a month). The unit is the page; clients matter only because their pages are correlated, which drives the validation design.

**Features — the complete list, the same one everywhere (paper, notebooks, receipts).** `log_recent30_impressions`, `recent30_ctr_pct`, `recent30_avg_position`, `recent30_active_days`, `content_age_days`. Nothing else reaches the models.

**Baseline.** The Week-4 hand rule: visible = at least 500 impressions; low CTR in top 10 = visible, position 1-10, CTR under 1%; stale = visible and 91+ days old. Score = 0.40*visible + 0.35*low_ctr_top10 + 0.25*stale, with five reason codes (`stale_low_ctr_top10`, `stale`, `low_ctr_top10`, `visible`, `low_visibility`) that later become the queue's reason column. The rule is a fair baseline because it encodes the two signals the Week-4 audit confirmed (91+ days: 55.0% vs 42.5% decline; top-10 low CTR: 50.3% vs 18.6%), and because it is evaluated under identical conditions as the models.

**Models.** Logistic regression (features standardized inside the training pipeline, so test-fold statistics never leak into training) and random forest. Library defaults, no tuning, seed 42 everywhere. Both output a probability; the probability is the ranking score.

**Validation design — the fair-test contract.** Model and baseline always see the same eligible rows, the same folds, the same metric (precision@50), and the same tie-break (impressions, highest first). Three tests, one question each: (A) a plain random split, run deliberately as the trap it is — how good does the model look when allowed to re-describe pages it has seen; (B) five client-grouped folds, zero clients shared per fold — can it rank pages for a client it has never seen; (C) fit March, score April, nothing retrained — does the ranking survive the month of drift between training and use, which is exactly how the queue is deployed. A fourth test, walk-forward with growing history, asks whether test C's drop is just a one-month-training artifact.

**Leakage checks.** Timeline drawn (features end strictly before the label window opens, asserted at build time for every frame). No label-derived features — the models see exactly the five contracted columns. No product flags — the rule score is a baseline to beat, never an input. IDs group, they do not learn. Population checked and its exclusions measured (Section 2). And the harness itself was tested: planting the label's own column lifted average precision from 0.761 to 0.993 — an audit that cannot catch a planted leak cannot clear an honest one. The column was removed; every reported number uses the honest five.


In [3]:
# The methodology receipts: fold composition (grouping proven), leak injection, tie bands.
folds = pd.DataFrame(model["fold_composition"])[
    ["fold", "train_rows", "test_rows", "train_clients", "test_clients", "client_overlap",
     "train_positive_rate", "test_positive_rate"]]
print("Five client-grouped folds (client_overlap must be 0 everywhere):")
display(folds)

lk = val["leak_injection"]
print(f"Planted-leak test: honest average precision {lk['honest_ap']:.3f} -> "
      f"with the label's own column planted in {lk['injected_ap']:.3f}")

print("Rule tie bands (pages sharing the maximum score, the model's job is to order them):")
for k in ("march", "april"):
    band = play["rule_tie_band"][k]
    print(f"  {k}: {band['n']:,} pages, decline rate {band['decline_rate']:.3f}")


Five client-grouped folds (client_overlap must be 0 everywhere):


,fold,train_rows,test_rows,train_clients,test_clients,client_overlap,train_positive_rate,test_positive_rate
0,1,71357,24453,30,10,0,0.447,0.620
1,2,71612,24198,30,10,0,0.522,0.398
2,3,28153,67657,30,10,0,0.558,0.463
3,4,69811,25999,30,10,0,0.529,0.389
4,5,70511,25299,30,10,0,0.457,0.585


Planted-leak test: honest average precision 0.761 -> with the label's own column planted in 0.993
Rule tie bands (pages sharing the maximum score, the model's job is to order them):
  march: 24,462 pages, decline rate 0.507
  april: 22,906 pages, decline rate 0.541


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

**Test B — the within-month win is real but wide.** Across five client-grouped folds, both models beat the rule at precision@50 on every fold: LR 0.636 (spread 0.195), RF 0.608 (0.230), rule 0.512 (0.201), against fold base rates of 0.39-0.62 (mean 0.49). The spread matters as much as the means: the rule wanders 0.32-0.84 because its top 50 is an arbitrary draw from the 24,462-page tie band; the models break those ties. Client clusters still move any single fold — the audit measured 829 rows with identical feature values in the March frame, and the strongest fold's top 50 came from just 3 of its 10 test clients. The paired LR-vs-RF difference (-0.028, sd 0.059) is inside fold noise: the two models are not separable on five folds, and LR is kept as the readable companion.

**Test A — the trap.** On a naive random split, RF scores 0.98 because 34 of the 40 clients sit on both sides: the model is re-describing pages it has already seen, not forecasting. Measured, reported, and treated as leakage, not skill.

**Test C — the edge is a one-cycle instrument.** Fit on March, scored on April: RF falls to 0.14 (April base rate 0.56), LR holds 0.54 — above the rule's 0.44 but not above picking at random. LR keeps some global ordering (average precision 0.624 vs the rule's 0.559), but the head of the list is what a 50-slot budget spends, and at the head the lift is gone. The base rate itself moved from about 0.20 in February to 0.56 in April — a moving target.

**Walk-forward — more history helps briefly, then stops.** On a fixed panel (the clients present in every frame, 20 clients per training side) with the test side always the same April rows: LR 0.56 -> 0.68 -> 0.70 -> 0.66 and RF 0.28 -> 0.46 -> 0.40 -> 0.48 as training history grows from one to four frames, while the rule reads 0.38 throughout. If the forward drop were only a one-month-training artifact the lines would climb with history; they peak around two to three frames and slide back. Refit monthly.

**What the models learned.** Permutation importance puts CTR first by a wide margin (shuffling it costs about 2.5x the next feature), then active days, then content age. LR's age coefficient came out negative even though staleness is a confirmed signal — because decline risk peaks at 90-180 days of age and falls after (April frame: 0.38 / 0.61 / 0.66 / 0.58 / 0.36 across the five age buckets), a shape one straight line cannot represent. The most confident mistakes repeated one pattern: rank 1-2, near-zero CTR (0.07-0.18%), thousands of impressions, often one client — usually a results page that answers the question itself or a brand-name search, which a rewrite cannot fix. That pattern is now a gate in the playbook.


In [4]:
# Tables 2-4 of the paper, built from the receipts.
g = val["grouped_cv"]["summary"]
table2 = pd.DataFrame({
    "test": ["A. random split", "B. client-grouped (5 folds)", "C. one month forward"],
    "rule P@50": [rand["metrics"]["baseline"]["p50"], g["baseline"]["p50_mean"], fwd["metrics"]["baseline"]["p50"]],
    "LR P@50": [rand["metrics"]["logistic_regression"]["p50"], g["logistic_regression"]["p50_mean"],
                fwd["metrics"]["logistic_regression"]["p50"]],
    "RF P@50": [rand["metrics"]["random_forest"]["p50"], g["random_forest"]["p50_mean"],
                fwd["metrics"]["random_forest"]["p50"]],
    "base rate": [rand["test_base_rate"],
                  sum(f["test_base_rate"] for f in val["grouped_cv"]["per_fold"]) / 5,
                  fwd["test_base_rate"]],
}).round(3)
print("Table 2 - precision@50 under the three tests:")
display(table2)

table3 = pd.DataFrame({
    "scorer": ["rule", "logistic regression", "random forest"],
    "P@50 mean": [g["baseline"]["p50_mean"], g["logistic_regression"]["p50_mean"], g["random_forest"]["p50_mean"]],
    "P@50 sd": [g["baseline"]["p50_sd"], g["logistic_regression"]["p50_sd"], g["random_forest"]["p50_sd"]],
    "AP mean": [g["baseline"]["ap_mean"], g["logistic_regression"]["ap_mean"], g["random_forest"]["ap_mean"]],
    "ROC-AUC mean": [g["baseline"]["roc_auc_mean"], g["logistic_regression"]["roc_auc_mean"],
                     g["random_forest"]["roc_auc_mean"]],
}).round(3)
print("Table 3 - client-grouped folds in detail (mean +/- sd across five folds):")
display(table3)

wf = val["walk_forward"]["origins"]
table4 = pd.DataFrame({
    "training history": [f"{o['train_frames']} frame(s) through {o['history_through']}" for o in wf],
    "rule P@50": [o["metrics"]["baseline"]["p50"] for o in wf],
    "LR P@50": [o["metrics"]["logistic_regression"]["p50"] for o in wf],
    "RF P@50": [o["metrics"]["random_forest"]["p50"] for o in wf],
})
print("Table 4 - walk-forward on the fixed April test (base rate "
      f"{wf[0]['test_base_rate']:.3f}, rule flat by construction):")
display(table4)


Table 2 - precision@50 under the three tests:


,test,rule P@50,LR P@50,RF P@50,base rate
0,A. random split,0.440,0.740,0.980,0.492
1,B. client-grouped (5 folds),0.512,0.636,0.608,0.491
2,C. one month forward,0.440,0.540,0.140,0.560


Table 3 - client-grouped folds in detail (mean +/- sd across five folds):


,scorer,P@50 mean,P@50 sd,AP mean,ROC-AUC mean
0,rule,0.512,0.201,0.492,0.492
1,logistic regression,0.636,0.195,0.557,0.605
2,random forest,0.608,0.230,0.561,0.604


Table 4 - walk-forward on the fixed April test (base rate 0.547, rule flat by construction):


,training history,rule P@50,LR P@50,RF P@50
0,1 frame(s) through nov,0.38,0.56,0.28
1,2 frame(s) through dec,0.38,0.68,0.46
2,3 frame(s) through jan,0.38,0.70,0.40
3,4 frame(s) through feb,0.38,0.66,0.48


## 5. Limitations

*What this work cannot claim.*

- **Not causal.** No page was experimentally changed. Nothing here shows that reviewing, refreshing, or rewriting a page affects its traffic in either direction. The queue orders risk; it does not promise that acting on it prevents anything.
- **Short shelf life.** Strong in the month it is fit, near chance one month forward, not rescued by more history. Every recommendation in Section 6 follows from this: refit monthly, never act on last month's queue.
- **The label partly measures measurement.** Week 6 measured a 96% decline rate for pages with only 14-20 days of tracking in the outcome month, against about 40% for pages covered all 30 days. Some "declines" are gaps in tracking, not drops in demand.
- **Survivorship.** 2.6% (March) and 3.6% (April) of candidates were dropped for thin or absent outcome tracking, and those lean toward pages going quiet.
- **Client concentration cuts both ways.** The largest client is 22% of a frame; 829 frame rows share identical feature values; and in the current cycle one client holds 48% of the budget queue against a 35% limit — the monitoring clock flagged it. Fold means are five draws of unequal, correlated data (fold 3's test side is 70.6% of the frame), not five independent experiments.
- **Five numbers cannot see content.** Quality, search intent, results-page layout, seasonality, and business value are invisible to the features. A human reviewer sees all of them; that division of labour is the point.
- **One development month.** The headline comparison lives on March 2026. The walk-forward covers more history but on a smaller fixed panel.
- **Untuned defaults.** Both models run library-default hyperparameters. Tuning might move the numbers; it would not change the shape of the story.

**The claim this work stands behind.** Observed across five client-grouped folds in March 2026, both learned models ranked the review queue above the hand rule (means 0.636 and 0.608 versus 0.512, base rate 0.49). Measured one month forward, the lift at the head of the queue mostly disappeared (0.54 versus a 0.56 base rate). The models are directional decision support for choosing which pages a person reviews — refit monthly, checked against the rule, never read as a forecast or a causal argument.


In [5]:
# The receipts behind the limitations section.
fa = val["fold4_anomaly"]
print("Client clusters (w06 receipt):")
print(f"  rows with identical feature values in the March frame: {fa['frame_duplicate_feature_vectors']}")
print(f"  strongest fold: top-50 rows came from {fa['top50_clients']} of 10 test clients;"
      f" {fa['rows_proba_gt_0.9']} frame rows carry a >0.9 model probability")
foldsz = model["fold_composition"]
print(f"  fold test sides range {min(f['test_rows'] for f in foldsz):,}-"
      f"{max(f['test_rows'] for f in foldsz):,} rows; fold 3 alone is "
      f"{max(f['test_rows'] for f in foldsz) / model['frame_rows']:.1%} of the frame")
print("\nQueue concentration this cycle (w07 receipt):")
q = play["queue"]
print(f"  {q['top50_clients']} clients fill the 50-row queue; the largest holds "
      f"{q['top50_largest_client_share']:.0%} (35% limit -> triggered)")
print("\nBase-rate drift across the panel (w06): about 0.20 in February -> "
      f"{val['frames']['april']['base_rate']:.2f} in April - a moving target for any fixed model.")


Client clusters (w06 receipt):
  rows with identical feature values in the March frame: 829
  strongest fold: top-50 rows came from 3 of 10 test clients; 112 frame rows carry a >0.9 model probability
  fold test sides range 24,198-67,657 rows; fold 3 alone is 70.6% of the frame

Queue concentration this cycle (w07 receipt):
  10 clients fill the 50-row queue; the largest holds 48% (35% limit -> triggered)

Base-rate drift across the panel (w06): about 0.20 in February -> 0.56 in April - a moving target for any fixed model.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

**The queue.** Built the way test C validated it: models fit on March, scoring April. **The rule explains, the model orders.** Each of the 50 rows carries the anonymous page code, the model probability (the order, tie-broken by impressions), the rule flag (the reason), one of four actions, and a refresh hint pointing at the kind of work the features suggest.

**Four actions.** `review_before_revert` — the default: look at the page and the searches that found it, write a hypothesis, then change content. `verify_then_review` — impressions moved more than 7x month-over-month: check tracking before blaming demand. `monitor_only` — impressions at least 1.5x last month: rising, do not touch (a one-month test, disclosed as such). `investigate_quiet_risk` — model probability 0.70 or higher with no rule flag: the model sees something it cannot explain, so a person checks.

**Gates before the budget cut.** New pages (under 30 days or no prior-month history) excluded; risers (1.5x) excluded; rank-1/2 near-zero-CTR pages get a senior look; extreme jumps (20x) get tracking verification; below-floor pages never entered the frame; a recently-edited lockout is proposed only — the release has no edit-history table. This cycle the gates removed 14 rows from the ungated top 50.

**What this cycle's queue actually was — the honest read.** All 50 rows are pages the rule does not flag: small pages (100-150 impressions) with exactly 0.0% CTR at visible positions. Observed against April labels, the ungated top 50 declined at 0.54 (base 0.56); after gates, 0.56 — exactly the base rate. This month the queue is a diagnostic list, not a refresh list: 49 quiet-risk investigations at about an hour each (about 49.5 editor-hours, about 4,700 impressions at stake). That is the model behaving as trained, and it is why the queue is framed as a review order, not a forecast.

**Ranked actions, in order.**

1. Cap per-client share of the queue before it goes to editors — one client holds 48% against a 35% limit. Until the cap is set, the list is not ready to hand out.
2. Use the queue as a review order with the checklist: what people searched for, what the results page looked like, technical state, seasonality, business value, sensitive-topic routing. Reviewer verdicts (`act / defer / not_actionable / escalate`) are the labels a future model should learn from.
3. Refit monthly; retire any queue older than its feature month.
4. Keep the fall-back armed: if the model's P@50 is below the rule's for two consecutive labelled months, order by the rule until a refit beats it. This cycle: LR 0.54 vs rule 0.44 — not triggered; the stricter base-rate check (clear the base by 0.05) did trigger, and is reported rather than hidden.
5. Watch the drift clocks: feature stability (position 0.108 and content age 0.179 in the PSI "watch" band, under the 0.25 act line), schema, base-rate drift — plus the post-release clock that labels last month's queue.
6. Fix the data gaps the gates exposed: an edit-history table would make the recently-edited lockout real.

**The no-go list — never automated, whatever the score.** No automated deletions, redirects, de-indexing or merges. No unreviewed AI-written rewrites. No automated edits to health, money, legal or safety pages. No bulk template changes triggered by page scores. No client-facing promises (P@50 varies by about 0.2 between test splits). No acting on `monitor_only` rows or rewriting rank-1/2 near-zero-CTR pages beyond checking which searches surface them.


In [6]:
# The playbook receipts: actions, gates, the queue, and the monitoring clocks.
ad = play["action_distribution_april"]
print("Action attached to each April page (before the budget cut):")
for k, v in ad.items():
    print(f"  {k:24s} {v:>7,}")

gh = play["gates"]["hits_april"]
print(f"\nGates (April): new pages {gh['NEW_PAGE']:,} | risers {gh['RISER_1M']:,} | "
      f"senior-look {gh['TOP2_NEAR_ZERO_CTR']:,} | verify {gh['EXTREME_JUMP_20X']:,}"
      f" -> eligible {play['gates']['eligible_rows']:,} of {april['rows']:,}")

q = play["queue"]
print(f"\nBudget queue (top 50 eligible): decline rate {q['top50_decline_rate_after_gates']:.2f} "
      f"(ungated head {q['top50_decline_rate_before_gates']:.2f}, April base {april['base_rate']:.2f}); "
      f"{q['top50_clients']} clients, largest {q['top50_largest_client_share']:.0%}; "
      f"actions: " + ", ".join(f"{k} {v}" for k, v in q["actions"].items()) + "; "
      f"rule flags: " + ", ".join(f"{k} {v}" for k, v in q["rule_flags"].items()))

print("\nMonitoring clocks, this cycle:")
for m in play["monitoring"]:
    print(f"  [{m['status']:9s}] {m['clock']}-release: {m['trigger']} = {m['value']} (threshold {m['threshold']})")


Action attached to each April page (before the budget cut):
  review_before_revert      83,619
  monitor_only              14,426
  verify_then_review         1,078
  investigate_quiet_risk       156

Gates (April): new pages 14,935 | risers 14,850 | senior-look 214 | verify 103 -> eligible 69,494 of 99,279

Budget queue (top 50 eligible): decline rate 0.56 (ungated head 0.54, April base 0.56); 10 clients, largest 48%; actions: investigate_quiet_risk 49, verify_then_review 1; rule flags: low_visibility 50

Monitoring clocks, this cycle:
  [PASS     ] pre-release: feature PSI (worst feature) = content_age_days 0.179 (threshold < 0.25)
  [PASS     ] pre-release: schema: five features present, no nulls = ok (threshold ok)
  [PASS     ] pre-release: base-rate drift (April vs March) = 0.069 (threshold <= 0.10)
  [TRIGGERED] pre-release: client concentration (top-50) = 48% (threshold <= 35%)
  [PASS     ] post-release: true base rate of the labelled month = 0.560 (threshold recorded)
  [PASS

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show — then ship it.*

The paper is a single self-contained file, `work/paper/index.html` — figures embedded, nothing of this project outside `work/`. The figure sources live in `work/figures/`, and the cell below regenerates them from the committed receipts and re-embeds them into the page (the same code lives in `work/scripts/make_paper_figures.py` for local runs):

- `paper_fig1_results.png` — the paper in one chart: the three scorers under the three tests (Table 2).
- `paper_fig2_walkforward.png` — more history helps briefly, then stops (Table 4).
- `paper_fig3_age_decay.png` — decline risk peaks at 3-6 months of age, then falls.
- `paper_fig4_playbook.png` — the funnel from April pages to the 50-row queue, and the action mix.
- `paper_fig5_leak_check.png` — the planted-leak test: the checks can catch a leak.

**Deploying the paper (a `gh-pages` branch carries the published page; `work/` carries everything else).**

`work/paper/index.html` is the source; deployment copies that one file to the root of a dedicated `gh-pages` branch, which GitHub Pages serves. `work/scripts/deploy_paper.py` does the copy, commit and push — it never touches the working tree or the current branch, and it refuses to publish if the page is missing the flyrank.ai credit or any of the five embedded figures:

1. Merge this work to `main`.
2. From the repo root: `python work/scripts/deploy_paper.py --dry-run` to preview, then `python work/scripts/deploy_paper.py` to publish.
3. Once, in GitHub: Settings -> Pages -> Source "Deploy from a branch" -> Branch `gh-pages`, folder `/ (root)` -> Save.
4. Wait a minute or two, then open the URL Settings -> Pages shows (for this repo: `https://<user>.github.io/FlyRank-ML-Internship/`). Verify in a private/incognito window and on a phone: the charts render and every link works — the charts are embedded in the file, so the page is one self-contained download.
5. Paste that exact URL into `submission/paper_url.txt` — one line, nothing else.
6. Run this notebook top to bottom and save it **with outputs before setting the URL**: once `paper_url.txt` holds a real URL, the repo's CI fails if a deliverable notebook still has unexecuted code cells. The final cell below re-checks every headline number against its receipt — the paper's numbers and the receipts must agree before anything ships.

After any later paper edit: rerun `make_paper_figures.py` (or this notebook) to rebuild `work/paper/index.html`, then rerun `deploy_paper.py`. Never edit files on `gh-pages` by hand — it is published output.

**Public-safety pass, done once more before submitting:** no client names, URLs, or private queries anywhere (the release has none and this notebook adds none); claim language stays observed / measured / directional / decision-support; the flyrank.ai data credit is in the paper's footer and acknowledgments.


In [7]:

# Regenerate the paper's five figures from the committed receipts.
# This is the SAME figure code as work/scripts/make_paper_figures.py, so rerunning
# this notebook reproduces the deployed page exactly.
import matplotlib
import matplotlib.pyplot as plt

C_RULE, C_LR, C_RF, C_BASE, C_TEXT = "#94a3b8", "#4338ca", "#0f766e", "#d97706", "#1e293b"
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "#cbd5e1", "axes.labelcolor": C_TEXT,
                     "text.color": C_TEXT, "xtick.color": "#475569", "ytick.color": "#475569",
                     "axes.grid": True, "grid.color": "#e2e8f0", "axes.axisbelow": True,
                     "grid.linewidth": 0.8})

FIG_DIR = os.path.join("work", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("wrote", os.path.join(FIG_DIR, name))

def label_bars(ax, bars, fmt="{:.2f}", dy=0.015):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + dy, fmt.format(b.get_height()),
                ha="center", va="bottom", fontsize=10, fontweight="bold", color=C_TEXT)

def strip(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

rand, grouped, fwd = val["random_split"], val["grouped_cv"]["summary"], val["time_forward"]
gbase = sum(f["test_base_rate"] for f in val["grouped_cv"]["per_fold"]) / 5

# Fig 1 - the three tests.
fig, axes = plt.subplots(1, 3, figsize=(11.5, 4.1), sharey=True)
panels = [("A. Random split\n(the trap)",
           [rand["metrics"]["baseline"]["p50"], rand["metrics"]["logistic_regression"]["p50"], rand["metrics"]["random_forest"]["p50"]],
           rand["test_base_rate"], "34 of 40 clients on both sides"),
          ("B. Clients kept separate\n(5 folds, mean \u00b1 spread)",
           [grouped["baseline"]["p50_mean"], grouped["logistic_regression"]["p50_mean"], grouped["random_forest"]["p50_mean"]],
           gbase, "0 clients shared per fold"),
          ("C. One month forward\n(fit March, score April)",
           [fwd["metrics"]["baseline"]["p50"], fwd["metrics"]["logistic_regression"]["p50"], fwd["metrics"]["random_forest"]["p50"]],
           fwd["test_base_rate"], "0.56 base rate in April")]
errs = [None,
        [grouped["baseline"]["p50_sd"], grouped["logistic_regression"]["p50_sd"], grouped["random_forest"]["p50_sd"]],
        None]
for ax, (title, vals_p, b, note), err in zip(axes, panels, errs):
    bars = ax.bar(range(3), vals_p, width=0.62, color=[C_RULE, C_LR, C_RF], edgecolor="white",
                  linewidth=1.2, yerr=err, error_kw=dict(ecolor=C_TEXT, lw=1.4, capsize=5, capthick=1.4))
    label_bars(ax, bars)
    ax.axhline(b, color=C_BASE, ls="--", lw=1.6)
    ax.text(2.42, b + 0.015, f"base rate\n{b:.2f}", color=C_BASE, fontsize=8.5, ha="right", fontweight="bold")
    ax.set_xticks(range(3)); ax.set_xticklabels(["Rule", "Logistic\nregression", "Random\nforest"], fontsize=9.5)
    ax.set_title(title, fontsize=11.5, fontweight="bold", pad=10); ax.set_ylim(0, 1.12)
    ax.text(0.5, -0.30, note, transform=ax.transAxes, ha="center", fontsize=9, color="#64748b")
    strip(ax)
axes[0].set_ylabel("Precision@50", fontweight="bold")
fig.suptitle("The same three scorers, three increasingly honest tests", fontsize=13.5, fontweight="bold", y=1.04)
fig.text(0.035, -0.055, "Precision@50 = share of the 50 highest-ranked pages that really declined next month. "
         "Bars above the dashed base-rate line beat picking pages at random.", fontsize=9.5, color="#64748b")
save(fig, "paper_fig1_results.png")

# Fig 2 - walk-forward.
wf = val["walk_forward"]["origins"]
depth = [o["train_frames"] for o in wf]
lr_p = [o["metrics"]["logistic_regression"]["p50"] for o in wf]
rf_p = [o["metrics"]["random_forest"]["p50"] for o in wf]
rule_p = [o["metrics"]["baseline"]["p50"] for o in wf]
wf_base = wf[0]["test_base_rate"]
fig, ax = plt.subplots(figsize=(8.6, 4.3))
ax.plot(depth, lr_p, "-o", color=C_LR, linewidth=2.4, markersize=7, label="Logistic regression")
ax.plot(depth, rf_p, "-o", color=C_RF, linewidth=2.4, markersize=7, label="Random forest")
ax.plot(depth, rule_p, "-s", color="#64748b", linewidth=2.0, markersize=6, label="Hand rule")
ax.axhline(wf_base, color=C_BASE, linestyle="--", linewidth=1.8)
ax.text(4.02, wf_base + 0.012, f"base rate {wf_base:.2f}", color=C_BASE, fontsize=9.5, fontweight="bold", ha="right")
for d, y in zip(depth, lr_p):
    ax.annotate(f"{y:.2f}", (d, y), textcoords="offset points", xytext=(0, 9), ha="center",
                fontsize=9, fontweight="bold", color=C_LR)
for d, y in zip(depth, rf_p):
    ax.annotate(f"{y:.2f}", (d, y), textcoords="offset points", xytext=(0, -16), ha="center",
                fontsize=9, fontweight="bold", color=C_RF)
ax.set_xticks(depth)
ax.set_xticklabels([f"{d} month{'s' if d > 1 else ''} of training pages" for d in depth], fontsize=9.5)
ax.set_xlabel("Training history (frames ending Feb, Jan, Dec, Nov 2025 -> March 2026 labels)", fontweight="bold")
ax.set_ylabel("Precision@50 on the same April test", fontweight="bold"); ax.set_ylim(0.2, 0.85)
ax.set_title("More history helps briefly, then stops helping", fontsize=13, fontweight="bold", pad=10)
ax.legend(loc="upper right", frameon=True, edgecolor="#e2e8f0", fontsize=9.5); strip(ax)
fig.text(0.01, -0.04, "Fixed panel: the clients present in every frame (20 clients per training side); the test side is "
         "the same April rows throughout. Rule flat at 0.38 by construction of the fixed test.",
         fontsize=9, color="#64748b")
save(fig, "paper_fig2_walkforward.png")

# Fig 3 - age decay.
b_ = play["age_bucket_decline_rate_april"]
names = [x["age_bucket"] for x in b_]; rates = [x["decline_rate"] for x in b_]
apr_base = play["frames"]["score_april"]["base_rate"]
fig, ax = plt.subplots(figsize=(8.6, 4.3))
bars = ax.bar(range(len(names)), rates, width=0.62,
              color=["#c7d2fe" if r != max(rates) else C_LR for r in rates], edgecolor="white", linewidth=1.2)
label_bars(ax, bars)
ax.axhline(apr_base, color=C_BASE, linestyle="--", linewidth=1.8)
ax.text(4.45, apr_base + 0.012, f"April base rate {apr_base:.2f}", color=C_BASE, fontsize=9.5, fontweight="bold", ha="right")
ax.set_xticks(range(len(names)))
ax.set_xticklabels([f"{n}\n({x['pages_pct']:.0f}% of pages)" for n, x in zip(names, b_)], fontsize=9.5)
ax.set_xlabel("Content age at the start of the month", fontweight="bold")
ax.set_ylabel("Share that declined next month", fontweight="bold"); ax.set_ylim(0, 0.78)
ax.set_title("Decline risk peaks at 3-6 months of age, then falls", fontsize=13, fontweight="bold", pad=10)
strip(ax)
fig.text(0.01, -0.04, "April 2026 frame (99,279 pages). A straight-line age term cannot represent this shape - "
         "which is why the linear model's age coefficient came out with the wrong sign.", fontsize=9, color="#64748b")
save(fig, "paper_fig3_age_decay.png")

# Fig 4 - the playbook funnel + action mix.
gh = play["gates"]["hits_april"]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax = axes[0]
stages = ["Pages in the\nApril frame", "Eligible after\ngates", "Budget queue\n(top 50)", "Really\ndeclined"]
nums = [april["rows"], play["gates"]["eligible_rows"], play["queue"]["size"],
        round(play["queue"]["top50_decline_rate_after_gates"] * play["queue"]["size"])]
yp = list(range(len(stages)))[::-1]
ax.barh(yp, nums, height=0.62, color=["#a5b4fc", "#818cf8", C_LR, "#312e81"], edgecolor="white", linewidth=1.2)
ax.set_xscale("log"); ax.set_yticks(yp); ax.set_yticklabels(stages, fontsize=9.5)
for y, n in zip(yp, nums):
    ax.text(n * 1.15, y, f"{n:,}", va="center", fontsize=10, fontweight="bold", color=C_TEXT)
gate_note = (f"gates removed: new pages {gh['NEW_PAGE']:,} - risers {gh['RISER_1M']:,} - "
             f"senior-look {gh['TOP2_NEAR_ZERO_CTR']:,} - verify {gh['EXTREME_JUMP_20X']:,}")
ax.set_title("From the month's pages to a 50-row queue", fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("pages (log scale)", fontweight="bold")
ax.text(0.5, -0.24, gate_note, transform=ax.transAxes, ha="center", fontsize=8.5, color="#64748b")
strip(ax)
ax = axes[1]
anames = list(ad.keys()); avals = [ad[k] for k in anames]
yp = list(range(len(anames)))[::-1]
ax.barh(yp, avals, height=0.62, color=["#818cf8", "#34d399", "#fbbf24", "#f87171"], edgecolor="white", linewidth=1.2)
ax.set_xscale("log"); ax.set_yticks(yp); ax.set_yticklabels([a.replace("_", "\n") for a in anames], fontsize=9)
for y, v in zip(yp, avals):
    ax.text(v * 1.15, y, f"{v:,}", va="center", fontsize=10, fontweight="bold", color=C_TEXT)
ax.set_title("The action attached to each April page", fontsize=12, fontweight="bold", pad=10)
ax.set_xlabel("pages (log scale)", fontweight="bold")
ax.text(0.5, -0.24, "Actions are assigned before the budget cut; only the top 50 eligible rows reach an editor.",
        transform=ax.transAxes, ha="center", fontsize=8.5, color="#64748b")
strip(ax)
fig.suptitle("What the playbook produced for May 2026", fontsize=13.5, fontweight="bold", y=1.03)
save(fig, "paper_fig4_playbook.png")

# Fig 5 - the planted-leak test.
lk = val["leak_injection"]
fig, ax = plt.subplots(figsize=(6.8, 3.5))
bars = ax.bar([0, 1], [lk["honest_ap"], lk["injected_ap"]], width=0.5, color=[C_RF, "#dc2626"], edgecolor="white", linewidth=1.2)
label_bars(ax, bars)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Honest five features", "With the label's own\ncolumn planted in"], fontsize=10)
ax.set_ylabel("Average precision (one fold)", fontweight="bold"); ax.set_ylim(0, 1.12)
ax.set_title("The check can catch a leak", fontsize=12.5, fontweight="bold", pad=10)
ax.annotate("0.99 = the score a leak buys.\nRemoved; every reported number\nuses the honest five.",
            xy=(1, lk["injected_ap"]), xytext=(0.32, 0.88), fontsize=9.5, color="#b91c1c",
            arrowprops=dict(arrowstyle="->", color="#b91c1c", lw=1.4))
strip(ax)
save(fig, "paper_fig5_leak_check.png")

# Re-embed the figures into the deployed page when running inside the repo.
# the page lives at work/paper/index.html and carries its figures inside it.
import base64
import re as _re
PAGE = os.path.join("work", "paper", "index.html")
if os.path.exists(PAGE):
    html = open(PAGE, encoding="utf-8").read()
    for name in ["paper_fig1_results.png", "paper_fig2_walkforward.png", "paper_fig3_age_decay.png",
                 "paper_fig4_playbook.png", "paper_fig5_leak_check.png"]:
        with open(os.path.join(FIG_DIR, name), "rb") as f:
            b64 = base64.b64encode(f.read()).decode("ascii")
        html, n = _re.subn(
            rf'<img[^>]*data-fig="{_re.escape(name)}"[^>]*>',
            lambda m, b64=b64: _re.sub(r'src="[^"]*"', f'src="data:image/png;base64,{b64}"', m.group(0), count=1),
            html, count=1)
        assert n == 1, f"figure {name} not found in the page"
    open(PAGE, "w", encoding="utf-8").write(html)
    print("re-embedded 5 figures into work/paper/index.html - the page needs no other files")
else:
    print("no work/paper/index.html here (e.g. Colab) - figures left in", FIG_DIR)


wrote work\figures\paper_fig1_results.png
wrote work\figures\paper_fig2_walkforward.png
wrote work\figures\paper_fig3_age_decay.png
wrote work\figures\paper_fig4_playbook.png
wrote work\figures\paper_fig5_leak_check.png
no work/paper/index.html here (e.g. Colab) - figures left in work\figures


In [8]:
# Receipt check: every headline number the paper quotes, re-checked against its receipt.
# If any line fails, the paper and the receipts disagree and something must be fixed before shipping.
checks = []

def check(label, got, want, tol=5e-4):
    if isinstance(got, list) and isinstance(want, list):
        ok = len(got) == len(want) and all(abs(a - b) <= tol for a, b in zip(got, want))
    elif isinstance(got, (int, float)) and isinstance(want, (int, float)):
        ok = abs(got - want) <= tol
    else:
        ok = got == want
    checks.append((label, got, want, ok))

# Table 2 - the three tests (P@50).
check("grouped rule P@50", g["baseline"]["p50_mean"], 0.512)
check("grouped LR P@50", g["logistic_regression"]["p50_mean"], 0.636)
check("grouped RF P@50", g["random_forest"]["p50_mean"], 0.608)
check("random rule P@50", rand["metrics"]["baseline"]["p50"], 0.44)
check("random LR P@50", rand["metrics"]["logistic_regression"]["p50"], 0.74)
check("random RF P@50 (the trap)", rand["metrics"]["random_forest"]["p50"], 0.98)
check("forward rule P@50", fwd["metrics"]["baseline"]["p50"], 0.44)
check("forward LR P@50", fwd["metrics"]["logistic_regression"]["p50"], 0.54)
check("forward RF P@50", fwd["metrics"]["random_forest"]["p50"], 0.14)

# Base rates and frames.
check("March base rate", val["frames"]["march"]["base_rate"], 0.4909)
check("April base rate", val["frames"]["april"]["base_rate"], 0.5596)
check("March pages", val["frames"]["march"]["rows"], 95810)
check("April pages", val["frames"]["april"]["rows"], 99279)
check("random-split shared clients", rand["shared_clients_both_sides"], 34)

# Walk-forward (LR 0.56/0.68/0.70/0.66, RF 0.28/0.46/0.40/0.48, rule flat 0.38).
check("walk LR", lr_p, [0.56, 0.68, 0.70, 0.66])
check("walk RF", rf_p, [0.28, 0.46, 0.40, 0.48])
check("walk rule", rule_p, [0.38] * 4)

# Leak injection.
check("leak honest AP", lk["honest_ap"], 0.761)
check("leak planted AP", lk["injected_ap"], 0.993)

# Tie band and age buckets.
check("March tie band size", base["tie_band"]["n"], 24462)
check("age-bucket decline rates", [x["decline_rate"] for x in b_],
      [0.3757, 0.6080, 0.6603, 0.5821, 0.3604], tol=1e-3)

# Playbook.
check("queue decline rate after gates", play["queue"]["top50_decline_rate_after_gates"], 0.56)
check("queue size", play["queue"]["size"], 50)
check("queue clients", play["queue"]["top50_clients"], 10)
check("queue largest client share", play["queue"]["top50_largest_client_share"], 0.48)
check("eligible rows after gates", play["gates"]["eligible_rows"], 69494)
check("quiet-risk rows in queue", play["queue"]["actions"]["investigate_quiet_risk"], 49)
check("monitoring triggers fired", len(play["monitoring_triggered"]), 2)
check("March survivorship drop share", (98398 - 95810) / 98398, 0.026, tol=5e-4)
check("April survivorship drop share", (102943 - 99279) / 102943, 0.036, tol=5e-4)
check("fold 3 share of frame", max(f["test_rows"] for f in model["fold_composition"]) / model["frame_rows"], 0.706, tol=5e-4)

out = pd.DataFrame(checks, columns=["check", "from receipt", "paper quotes", "ok"])
display(out)
assert out["ok"].all(), "A paper number disagrees with its receipt - fix before shipping."
print(f"ALL {len(out)} CHECKS PASS - the paper's numbers match the committed receipts.")
print("Run versions (the record, not an assertion):", val["library_versions"])


,check,from receipt,paper quotes,ok
0,grouped rule P@50,0.512,0.512,True
1,grouped LR P@50,0.636,0.636,True
2,grouped RF P@50,0.608,0.608,True
3,random rule P@50,0.44,0.44,True
4,random LR P@50,0.74,0.74,True
5,random RF P@50 (the trap),0.98,0.98,True
6,forward rule P@50,0.44,0.44,True
7,forward LR P@50,0.54,0.54,True
8,forward RF P@50,0.14,0.14,True
9,March base rate,0.49092,0.4909,True


ALL 31 CHECKS PASS - the paper's numbers match the committed receipts.
Run versions (the record, not an assertion): {'pandas': '3.0.3', 'numpy': '2.5.1', 'scikit-learn': '1.9.0', 'duckdb': '1.5.4'}


## Self-check

Before submitting, confirm each line honestly (tick these **after** running the notebook top to bottom):

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) — save it **with outputs** before setting the paper URL
- [ ] The receipt check in Section 7 prints ALL CHECKS PASS
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] `submission/paper_url.txt` holds the exact deployed URL — one line, nothing else.
- [x] ML-12 (5-minute demo outline, social post, employer summary) still to be done in this notebook's closing cells — separate card, not forgotten.
